In [0]:
-- Baseline 1: total revenue and trip count on the settled Silver table
SELECT
  ROUND(SUM(CAST(total_amount AS DECIMAL(12,2))), 2) AS total_january_revenue,
  COUNT(*)                                           AS total_trips
FROM nyc_mobility.silver.trips;

-- Baseline 2: distinct pickup dates
SELECT COUNT(DISTINCT DATE(tpep_pickup_datetime)) AS days_in_dataset
FROM nyc_mobility.silver.trips;

-- Baseline 3: pickup borough split
SELECT
  pickup_borough,
  COUNT(*)                                           AS trip_count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_trips
FROM nyc_mobility.silver.trips
GROUP BY pickup_borough
ORDER BY trip_count DESC;

-- Baseline 4: weekday versus weekend trip split
SELECT
  is_weekend,
  COUNT(*)                                           AS trip_count,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_trips
FROM nyc_mobility.silver.trips
GROUP BY is_weekend
ORDER BY is_weekend;


CREATE SCHEMA IF NOT EXISTS nyc_mobility.gold_expected;
-- Expected: revenue by date
CREATE OR REPLACE TABLE nyc_mobility.gold_expected.revenue_by_date AS
SELECT
  DATE(tpep_pickup_datetime)                         AS pickup_date,
  ROUND(SUM(CAST(total_amount AS DECIMAL(12,2))), 2) AS total_revenue,
  COUNT(*)                                           AS trip_count
FROM nyc_mobility.silver.trips
GROUP BY DATE(tpep_pickup_datetime);
-- Expected: trips by borough
CREATE OR REPLACE TABLE nyc_mobility.gold_expected.trips_by_borough AS
SELECT
  pickup_borough,
  COUNT(*)                                           AS trip_count,
  ROUND(SUM(CAST(total_amount AS DECIMAL(12,2))), 2) AS total_revenue
FROM nyc_mobility.silver.trips
GROUP BY pickup_borough;
-- Expected: trips by hour and weekend
CREATE OR REPLACE TABLE nyc_mobility.gold_expected.trips_by_hour_weekend AS
SELECT
  pickup_hour,
  is_weekend,
  COUNT(*) AS trip_count
FROM nyc_mobility.silver.trips
GROUP BY pickup_hour, is_weekend;

-- SELECT 'revenue_by_date'       AS table_name, COUNT(*) AS rows FROM nyc_mobility.gold_expected.revenue_by_date
-- UNION ALL
-- SELECT 'trips_by_borough',     COUNT(*) FROM nyc_mobility.gold_expected.trips_by_borough
-- UNION ALL
-- SELECT 'trips_by_hour_weekend', COUNT(*) FROM nyc_mobility.gold_expected.trips_by_hour_weekend;
